# Imports

In [26]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import wandb
import copy
from tabulate import tabulate
import tqdm
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
import odds_datasets
import warnings
warnings.filterwarnings('ignore')
# from scipy.stats import norm
import pandas as pd

from diffi.utils import *
from sklearn.ensemble import IsolationForest
from sklearn.metrics import f1_score, average_precision_score

Setting up W&B

In [2]:
wandb.login()

wandb: Currently logged in as: sanson-sebastiano-00 (sanson-sebastiano-00-universita-di-padova) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

Setting a seed value for reproducibility

In [3]:
np.random.seed(0)

# Helper functions

## Logging 

In [4]:
def log_feature_importance(feature_importances, threshold_type, og_model: bool):
    """
    Log feature importance plot to Weights & Biases.

    Args:
        feature_importances (np.ndarray): Array of feature importances.
        threshold_type (str): Type of thresholding used for feature selection.
        og_model (bool): Flag indicating if the model is original or pruned.
    """

    sorted_indices = np.argsort(feature_importances)[::-1]
    fi_std = np.std(feature_importances)

    plt.figure(figsize=(10, 5))
    plt.grid(True, axis='y', linestyle='--', alpha=0.7, zorder=0)
    plt.bar(range(len(sorted_indices)), feature_importances[sorted_indices], 
            yerr=fi_std, zorder=3)
    plt.xticks(range(len(sorted_indices)), sorted_indices)
    plt.xlabel('Feature Index')
    plt.ylabel('Feature Importance')
    plt.ylim(bottom=0)
    if og_model:
        plt.title('Feature Importance - Original Model')
        # wandb.log({f"threshold_type_{threshold_type}/feature_importance_original": wandb.Image(plt)})
    else:
        plt.title('Feature Importance - Pruned Model')
        # wandb.log({f"threshold_type_{threshold_type}/feature_importance_pruned": wandb.Image(plt)})
    # plt.close()
    plt.show()

In [5]:
def log_fi_heatmap(fis_in, fis_out, threshold_type, seed, is_og_model: bool):
    """
    Log feature importance heatmaps to wandb with inliers and outliers side-by-side.
    
    Args:
        fis_in (list): List of feature importance matrices for inliers, one per forest.
        fis_out (list): List of feature importance matrices for outliers, one per forest.
        threshold_type (str): Type of thresholding used for feature selection.
    """
    num_forests = len(fis_in)
    assert len(fis_out) == num_forests, "Number of forests doesn't match between inliers and outliers"
    
    for i in range(num_forests):
        # Check if the matrices are empty or have zero dimensions
        if (isinstance(fis_in[i], np.ndarray) and (fis_in[i].size == 0 or fis_in[i].shape[0] == 0)) or \
           (isinstance(fis_out[i], np.ndarray) and (fis_out[i].size == 0 or fis_out[i].shape[0] == 0)):
            print(f"Warning: Empty feature importance matrix for forest {i+1}. Skipping visualization.")
            wandb.log({f"threshold_type_{threshold_type}/seed_{seed}/warning": 
                      f"Empty feature importance matrix for forest {i+1} - visualization skipped"})
            continue

        fig, axes = plt.subplots(1, 2, figsize=(20, 8))
        
        # Inliers heatmap
        sns.heatmap(fis_in[i], cmap='viridis', cbar=True, vmin=0, vmax=0.16, ax=axes[0])
        if is_og_model:
            axes[0].set_title(f'Original Model Feature Importance - Inliers (Forest {i+1})', fontsize=14)
        else:
            axes[0].set_title(f'Pruned Model Feature Importance - Inliers (Forest {i+1})', fontsize=14)
        axes[0].set_xlabel('Feature Index')
        axes[0].set_ylabel('Tree Index')
        
        # Outliers heatmap 
        sns.heatmap(fis_out[i], cmap='viridis', cbar=True, vmin=0, vmax=0.16, ax=axes[1])
        if is_og_model:
            axes[1].set_title(f'Original Model Feature Importance - Outliers (Forest {i+1})', fontsize=14)
        else:   
            axes[1].set_title(f'Pruned Model Feature Importance - Outliers (Forest {i+1})', fontsize=14)
        axes[1].set_xlabel('Feature Index')
        axes[1].set_ylabel('Tree Index')
        
        plt.tight_layout()
        
        if is_og_model:
            wandb.log({f"threshold_type_{threshold_type}/seed_{seed}/feature_importance_heatmap_original": wandb.Image(fig)})
        else:
            wandb.log({f"threshold_type_{threshold_type}/seed_{seed}/feature_importance_heatmap_pruned": wandb.Image(fig)})
        
        plt.close(fig)

In [6]:
def log_fi_diff(og_fi, pruned_fi, threshold_type):
    """
    Log the difference in features importance between the original and pruned model.
    
    Args:
        og_fi (np.ndarray): Feature importance of the original model.
        pruned_fi (np.ndarray): Feature importance of the pruned model.
        threshold_type (str): Type of thresholding used for feature selection.
    """
    fi_diff = pruned_fi - og_fi
    colors = ['green' if val > 0 else 'red' for val in fi_diff]

    plt.figure(figsize=(10, 5))
    plt.grid(True, axis='y', linestyle='--', alpha=0.7, zorder=0)
    plt.bar(range(len(fi_diff)), fi_diff, color=colors, zorder=3)
    plt.xticks(range(len(fi_diff)), range(len(fi_diff)))
    plt.xlabel('Feature Index')
    plt.ylabel('Change in Feature Importance (Pruned - Original)')
    plt.title('Difference in Feature Importance After Pruning')

    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='green', edgecolor='black', label='Increase after pruning'),
        Patch(facecolor='red', edgecolor='black', label='Decrease after pruning')
    ]
    plt.legend(handles=legend_elements, loc='upper right')

    wandb.log({f"threshold_type_{threshold_type}/feature_importance_difference": wandb.Image(plt)})
    plt.close()

In [7]:
def log_estimators_summary(pruned_num_trees, og_num_trees, threshold_type, seed):
    """
    Log the number of estimators in each Isolation Forest to wandb.

    Args:
        iforests (list): List of Isolation Forest models.
        seed (int): Random seed used for the models.
        og_num_trees (int): Original number of trees in the unpruned model.
        threshold_type (str): Type of thresholding used for feature selection.
    """

    # # Log per-forest statistics
    # for i, iforest in enumerate(iforests):
    #     forest_count = len(iforest.estimators_)
    #     forest_index = i + 1  # Use 1-based indexing for readability
        
    #     wandb.log({
    #         f"threshold_type_{threshold_type}/seed_{seed}/forest_{forest_index}/num_estimators": forest_count,
    #         # Percentage of trees retained compared to original 
    #         f"threshold_type_{threshold_type}/seed_{seed}/forest_{forest_index}/retention_rate": forest_count / og_num_trees,
    #         f"threshold_type_{threshold_type}/seed_{seed}/forest_{forest_index}/mean_estimators": np.mean(forest_count),
    #         f"threshold_type_{threshold_type}/seed_{seed}/forest_{forest_index}/min_estimators": np.min(forest_count),
    #         f"threshold_type_{threshold_type}/seed_{seed}/forest_{forest_index}/max_estimators": np.max(forest_count),
    #     })

    # wandb.log({
    #     f"threshold_type_{threshold_type}/seed_{seed}/overall_total_estimators": np.sum([len(iforest.estimators_) for iforest in iforests]),
    # })

    # Log summary table
    data = [[i+1, pruned_num_trees[i]] for i in range(len(pruned_num_trees))]
    columns = ["Forest Index", "Number of Estimators"]
    wandb.log({f"threshold_type_{threshold_type}/seed_{seed}/estimators_table": wandb.Table(data=data, columns=columns)})

In [8]:
def log_features_summary(features_importance, selected_features, seed, threshold_type):
    """
    Log a summary of feature importances and selected meaningful features to wandb.
    Args:
        features_importance (np.ndarray): Array of feature importances.
        selected_features (list): List of indices of selected meaningful features for each forest.
        threshold_type (str): Type of thresholding used for feature selection.
    """

    for f in range(len(features_importance)):
        num_all_features = len(features_importance[f])
        num_meaningful_features = len(selected_features[f])
        # plot bar chart for each forest, highlighting selected features
        plt.figure(figsize=(10, 5))
        plt.grid(True, axis='y', linestyle='--', alpha=0.7, zorder=0)
        bars = plt.bar(range(len(features_importance[f])), features_importance[f], zorder=3)
        plt.xticks(range(len(features_importance[f])), range(len(features_importance[f])))
        plt.xlabel('Feature Index')
        plt.ylabel('Feature Importance')
        plt.ylim(bottom=0)
        for idx in selected_features[f]:
            bars[idx].set_color('orange')
        plt.legend(['Selected Features'], loc='upper right')
        plt.text(0.981, 0.88, f'Selected {num_meaningful_features}/{num_all_features} meaningful features', 
                 horizontalalignment='right', verticalalignment='top', transform=plt.gca().transAxes,
                 bbox=dict(facecolor='white', alpha=0.8, edgecolor='black'))
        plt.title(f'Feature Importance - Forest {f+1} (Seed {seed})')
        wandb.log({f"threshold_type_{threshold_type}/seed_{seed}/meaningful_features_selected": wandb.Image(plt)})
        plt.close()

In [9]:
def log_depth_vs_usage(num_forests, num_trees, unique_usages, avg_outliers_depth_mat, threshold_type, seed, isTrainingSet: bool):
    """
    Log bar plots comparing unique usages and ratio between average outliers and average inliers depth for each tree in each forest.
    Args:
        num_forests (int): Number of forests.
        num_trees (int): Number of trees in each forest.
        unique_usages (list): List of lists containing unique usages for each tree in each forest.
        avg_outliers_depth_mat (list): List of lists containing average outliers depth for each tree in each forest.
        threshold_type (str): Type of thresholding used for feature selection.
        seed (int): Random seed used for the models.
    """ 
    outlier_label = "Avg Training Outliers Depth" if isTrainingSet else "Avg Prediction Outliers Depth"

    for f in range(num_forests):
        plt.figure(figsize=(15, 7))
        width = 0.35  # the width of the bars
        x = np.arange(num_trees)  # the label locations

        plt.bar(x - width/2, unique_usages[f], width, label='Unique Usages', color='blue', alpha=0.7)
        plt.bar(x + width/2, avg_outliers_depth_mat[f], width, label=outlier_label, color='orange', alpha=0.7)   
        plt.xlabel('Tree Index')
        plt.ylabel('Value')
        plt.title(f'Comparison of Unique Usages and {outlier_label} (Seed {seed}, Forest {f+1})')
        plt.legend()
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.tight_layout()
        wandb.log({f"threshold_type_{threshold_type}/seed_{seed}/unique_usages_vs_avg_outliers_depth": wandb.Image(plt)})
        plt.close()

## Usage feature counter

In [10]:
def get_feature_usage(used_features, unique_features):
    """
    Calculating the feature usage in each tree for each forest.

    Args: 
        used_features (list): 
            - each element represents a forest and it is a list
                - each element of a forest is a tree and contains a list of features used in that tree
        unique_features (list): list of unique features used in the dataset.
    Returns:
        usage (np.ndarray): shape (num_forests, num_trees, num_features) 
            Each element is the count of how many times a feature is used in a tree of a forest.
        normalized_usage (np.ndarray): shape (num_forests, num_trees, num_features)
            Each element is the normalized count of how many times a feature is used in a tree of a forest. Normalization
            wrt to the total splits used in the tree. 
    """

    num_forests = len(used_features)
    num_trees = len(used_features[0])
    num_features = len(unique_features)

    usage = np.zeros((num_forests, num_trees, num_features), dtype=float)
    normalized_usage = np.zeros((num_forests, num_trees, num_features), dtype=float)

    # Iterate over each forest
    for f in range(num_forests):
        # Iterate over each tree in the forest
        for t in range(num_trees):
            internal_nodes_counter = 0
            # Iterate over each feature used in the tree
            for feature in used_features[f][t]:
                if feature in unique_features:
                    feature_index = unique_features.index(feature)
                    usage[f, t, feature_index] += 1
                if feature != -2:  # not a leaf node
                    internal_nodes_counter += 1
            
            # Normalize the usage by the number of internal nodes in the tree
            if internal_nodes_counter > 0:
                normalized_usage[f, t] = usage[f, t] / internal_nodes_counter
            else:
                raise ValueError(f"Tree {t} in forest {f} has no internal nodes, cannot normalize usage.")

    return usage, normalized_usage

## Meaningful feature selection methods

In [11]:
def elbow_features_selection(feature_importances):
    """
    Selecting most important features based on the elbow method.

    Args:
        feature_importances (np.ndarray): Array of shape (num_forests, num_features)
            containing the feature importances for each forest.

    Returns:
        selected_features (list): List of lists, where each inner list contains the indices of the selected features for each forest.
        threshold_gaps (np.ndarray): Array of shape (num_forests,) containing the maximum gap thresholds for each forest.
        gaps (list): List of lists, where each inner list contains the gaps between consecutive feature importances for each forest.
    """

    n_forests, n_features = feature_importances.shape

    # Get the indexes of the features sorted by importance
    sorted_indices = np.argsort(feature_importances, axis=1)[:, ::-1]

    # Sort the feature importances based on the sorted indices
    sorted_importances = np.take_along_axis(feature_importances, sorted_indices, axis=1)

    selected_features, gaps = [], []
    threshold_gaps = np.zeros(n_forests, dtype=float)

    if n_features < 2: # Not enough features to compute gaps
        for f in range(n_forests):
            selected_idx = sorted_indices[f, :].tolist()
            selected_features.append(selected_idx)
            gaps.append([])
    else:
        # Calculate gaps between consecutive feature importances
        gaps_batch = sorted_importances[:, :-1] - sorted_importances[:, 1:]

        # Get the maximum gap for each forest
        max_gap_indices = np.argmax(gaps_batch + 1e-12 * np.random.rand(*gaps_batch.shape), axis=1)
        threshold_gaps = np.take_along_axis(gaps_batch, max_gap_indices[:, np.newaxis], axis=1).squeeze()

        # Select features based on the maximum gap
        for f in range(n_forests):
            num_to_select = max_gap_indices[f] + 1
            selected_idx_for_forest = sorted_indices[f, :num_to_select].tolist()
            selected_features.append(selected_idx_for_forest)
            gaps.append(gaps_batch[f, :].tolist())

    return selected_features, threshold_gaps, gaps

## Selection of trees to be removed

In [12]:
def get_unique_usage_per_tree(normalized_usages, feature_importances, selected_features):
    """
    Compute the unique usage of selected features for each tree in each forest as weighted sum of features usage weighted by their importance.

    Args:
        normalized_usages (np.ndarray): Array of shape (num_forests, num_trees, num_features)
            containing the normalized feature usage for each tree in each forest.
        feature_importances (np.ndarray): Array of shape (num_forests, num_features)
            containing the feature importances for each forest.
        selected_features (list): List of lists, where each inner list contains the indices of the selected features for each forest.

    Returns:
        unique_usages (np.ndarray): Array of shape (num_forests, num_trees) containing the weighted usage for each tree in each forest.
    """

    n_forests, n_trees, n_features = normalized_usages.shape
    unique_usages = np.zeros((n_forests, n_trees), dtype=float)

    for f in range(n_forests):
        forest_selected_features = set(selected_features[f])
        forest_total_importance = np.sum(feature_importances[f, :])
        assert forest_total_importance > 0, f"Forest {f} has zero total importance, cannot compute unique usage."

        for t in range(n_trees):
            current_tree_weighted_usage = 0.0
            current_tree_weight_sum = 0.0

            for feature in range(n_features):
                if feature in forest_selected_features:
                    current_tree_weighted_usage += normalized_usages[f, t, feature] * feature_importances[f, feature]
                    current_tree_weight_sum += feature_importances[f, feature]

            assert current_tree_weight_sum > 0, f"Tree {t} in forest {f} has zero total importance, cannot compute unique usage."
            unique_usages[f, t] = current_tree_weighted_usage / current_tree_weight_sum
            
    return unique_usages

In [13]:
def get_unique_avg_usage_per_forest(normalized_usages, selected_features):
    """
    Compute the average usage of selected features across all trees in each forest.

    Args:
        normalized_usages (np.ndarray): Array of shape (num_forests, num_trees, num_features)
            containing the normalized feature usage for each tree in each forest.
        selected_features (list): List of lists, where each inner list contains the indices of the selected features for each forest.

    Returns:
        unique_avg_usage (np.ndarray): Array of shape (num_forests, num_features) containing the average usage of selected feature for each forest.
    """

    n_forests, _, n_features = normalized_usages.shape
    unique_avg_usage = np.zeros((n_forests, n_features), dtype=float)

    for f in range(n_forests):
        forest_selected_features = set(selected_features[f])

        for feature in range(n_features):
            if feature in forest_selected_features:
                unique_avg_usage[f, feature] = np.mean(normalized_usages[f, :, feature])

    return unique_avg_usage

def majority_voting(normalized_usages, selected_features, feature_importances):
    """ 
    Apply majority voting to compute weighted voting score 
    Args:
        normalized_usages (np.ndarray): Array of shape (num_forests, num_trees, num_features)
            containing the normalized feature usage for each tree in each forest.
        selected_features (list): List of lists, where each inner list contains the indices of the selected features for each forest.
        feature_importances (np.ndarray): Array of shape (num_forests, num_features)
            containing the feature importances for each forest.
    Returns:
        voting_scores (np.ndarray): Array of shape (num_forests, num_trees) containing the voting score for each tree in each forest.
    """
    
    n_forests, n_trees, n_features = normalized_usages.shape
    binary_scores = np.zeros((n_forests, n_trees, n_features), dtype=int)
    voting_scores = np.zeros((n_forests, n_trees), dtype=float)

    unique_avg_usage = get_unique_avg_usage_per_forest(normalized_usages, selected_features)

    for f in range(n_forests):
        forest_selected_features = set(selected_features[f])
        forest_total_importance = np.sum(feature_importances[f, :])
        assert forest_total_importance > 0, f"Forest {f} has zero total importance, cannot compute voting score."

        for t in range(n_trees):
            current_tree_weight_sum = 0.0

            for feature in range(n_features):
                if feature in forest_selected_features:
                    if normalized_usages[f, t, feature] > unique_avg_usage[f, feature]:
                        binary_scores[f, t, feature] = 1
                    current_tree_weight_sum += feature_importances[f, feature]

            assert current_tree_weight_sum > 0, f"Tree {t} in forest {f} has zero total importance, cannot compute voting score."
            voting_scores[f, t] = np.sum(np.multiply(binary_scores[f, t, :], feature_importances[f, :])) 

    return voting_scores

In [14]:
def remove_trees(iforests, unique_usages, thresholds):
    """
    For each forest, remove trees that have a feature usage below a certain threshold.

    Args:
        iforests (list): List of forests, each containing a list of trees.
        unique_usages (np.ndarray): Array of shape (num_forests, num_trees, num_features) 
            containing the usage of each feature in each tree.
        thresholds (list): List of thresholds for each forest to determine which trees to remove.

    Returns:
        pruned_forests (list): List of pruned forests with trees removed based on the thresholds.
    """
    pruned_forests = copy.deepcopy(iforests)

    n_forests = len(iforests)
    keep_mask = unique_usages > thresholds[:, np.newaxis]
    best_trees_idx = []

    for f in range(n_forests):
        # Get the index of the forest to prune
        current_forest = pruned_forests[f]
        current_mask = keep_mask[f]

        # Ensure at least one tree remains
        if not np.any(current_mask):
            # Keep the tree with highest usage
            best_tree_idx = np.argmax(unique_usages[f])
            current_mask[best_tree_idx] = True
            print(f"Warning: All trees would be pruned in forest {f}. Keeping the best tree.")

        og_estimators = current_forest.estimators_
        og_features = current_forest.estimators_features_

        # best_trees_idx.append([idx for idx, est in enumerate(og_estimators) if current_mask[idx]])

        # Filter and update the estimators and features based on the mask
        pruned_forests[f].estimators_ = [est for idx, est in enumerate(og_estimators) if current_mask[idx]]
        pruned_forests[f].estimators_features_ = [feat for idx, feat in enumerate(og_features) if current_mask[idx]]

        # Update also the internal attributes of the forest
        og_decision_paths = current_forest._decision_path_lengths
        pruned_forests[f]._decision_path_lengths = [path for idx, path in enumerate(og_decision_paths) if current_mask[idx]]

        og_avg_paths = current_forest._average_path_length_per_tree
        pruned_forests[f]._average_path_length_per_tree = [avg_path for idx, avg_path in enumerate(og_avg_paths) if current_mask[idx]]
        pruned_forests[f].n_estimators_ = len(pruned_forests[f].estimators_)
        
    return pruned_forests


In [15]:
def random_trees_removal(iforests, num_trees_to_remove, seed):
    """
    For each forest, remove trees that have a feature usage below a certain threshold.

    Args:
        iforests (list): List of forests, each containing a list of trees.
        unique_usages (np.ndarray): Array of shape (num_forests, num_trees, num_features) 
            containing the usage of each feature in each tree.
        thresholds (list): List of thresholds for each forest to determine which trees to remove.

    Returns:
        pruned_forests (list): List of pruned forests with trees removed based on the thresholds.
    """
    pruned_forests = copy.deepcopy(iforests)

    n_forests = len(iforests)

    for f in range(n_forests):
        # Get the index of the forest to prune
        current_forest = pruned_forests[f]
        n_trees = len(current_forest.estimators_)

        if num_trees_to_remove >= n_trees:
            raise ValueError(f"Cannot remove {num_trees_to_remove} trees from forest {f} with only {n_trees} trees.")
        
        # Mask of trees to keep (True) and remove (False)
        keep_mask = np.ones(n_trees, dtype=bool)
        rnd = np.random.RandomState(seed=seed + f)  
        indices_to_remove = rnd.choice(n_trees, num_trees_to_remove, replace=False)
        keep_mask[indices_to_remove] = False

        og_estimators = current_forest.estimators_
        og_features = current_forest.estimators_features_

        # Filter and update the estimators and features based on the mask
        pruned_forests[f].estimators_ = [est for idx, est in enumerate(og_estimators) if keep_mask[idx]]
        pruned_forests[f].estimators_features_ = [feat for idx, feat in enumerate(og_features) if keep_mask[idx]]

        # Update also the internal attributes of the forest
        og_decision_paths = current_forest._decision_path_lengths
        pruned_forests[f]._decision_path_lengths = [path for idx, path in enumerate(og_decision_paths) if keep_mask[idx]]

        og_avg_paths = current_forest._average_path_length_per_tree
        pruned_forests[f]._average_path_length_per_tree = [avg_path for idx, avg_path in enumerate(og_avg_paths) if keep_mask[idx]]
        pruned_forests[f].n_estimators_ = len(pruned_forests[f].estimators_)
        
    return pruned_forests


## Depth stats

In [16]:
def get_depth_stats(iforests, X_train):
    """
    Get depth statistics for each forest.

    Args:
        iforests (list): List of Isolation Forest models.
        X_train (np.ndarray): Training data.
        y (np.ndarray): Training or predicted labels
    Returns:
        depth (list): List of depth statistics for outliers and inliers for each forest.
    """
    depth_stats = []

    # For each forest
    for i in range(len(iforests)):
        current_iforest = iforests[i]

        tot_samples = X_train.shape[0]

        num_trees = len(current_iforest.estimators_)

        results = np.zeros((num_trees, tot_samples), dtype=object)

        # For each tree
        for j in range(num_trees):
            current_estimator = current_iforest.estimators_[j]
            
            # Get the leaf indices for each sample
            on_leaf = current_estimator.apply(X_train)
            
            # Get node depths
            node_depths = current_estimator.tree_.compute_node_depths()
            
            # For each sample
            for o in range(tot_samples):
                leaf_idx = on_leaf[o]
                if leaf_idx < len(node_depths) and current_estimator.tree_.children_left[leaf_idx] == -1:
                    depth = node_depths[leaf_idx]
                    if depth > 0:
                        results[j, o] = 1/depth
        
        depth_stats.append(results)

    return np.array(depth_stats)

In [17]:
def depth_based_pruning(iforests, depth_train, y_train, mode: str = 'mean', percentile=70, metric='separation_ratio'):
    """
    Prune trees based on their effectiveness at separating anomalies from normal samples.
    
    Args:
        iforests: List of Isolation Forest models
        depth_train: Array of depth statistics (shape: [num_forests, num_trees, num_samples])
        y_train: Training labels (0 for inliers, 1 for outliers)
        mode: 'mean' or 'percentile' to determine thresholding method
        percentile: Percentile threshold for tree removal
        metric: Separation metric ('separation_ratio', 'separation_diff', 'anomaly_depth')
    
    Returns:
        pruned_forests: List of pruned Isolation Forest models
    """
    num_forests = len(iforests)
    num_trees = len(iforests[0].estimators_)
    pruned_forests = copy.deepcopy(iforests)
    
    trees_score = np.zeros((num_forests, num_trees))
    
    for f in range(num_forests):
        for t in range(len(iforests[f].estimators_)):
            # Extract depth values (skip None values)
            tree_depths = depth_train[f][t]
            valid_indices = np.array([i for i in range(len(tree_depths)) if tree_depths[i] is not None])
            
            if len(valid_indices) > 0:
                valid_depths = np.array([tree_depths[i] for i in valid_indices], dtype=float)
                valid_labels = y_train[valid_indices]
                
                # Check both classes presence
                if np.sum(valid_labels) > 0 and np.sum(valid_labels == 0) > 0:
                    outlier_depths = valid_depths[valid_labels == 1]
                    inlier_depths = valid_depths[valid_labels == 0]
                    
                    # Averaging 
                    avg_outlier_depth = np.mean(outlier_depths) if len(outlier_depths) > 0 else 0
                    avg_inlier_depth = np.mean(inlier_depths) if len(inlier_depths) > 0 else 1e-10

                    # Extremes
                    # max_outlier_depth = np.min(outlier_depths) if len(outlier_depths) > 0 else 0
                    # min_inlier_depth = np.max(inlier_depths) if len(inlier_depths) > 0 else 1e-10
                    
                    if metric == 'separation_ratio':
                        trees_score[f, t] = avg_outlier_depth / (avg_inlier_depth + 1e-10)
                    elif metric == 'separation_diff':
                        trees_score[f, t] = avg_outlier_depth - avg_inlier_depth
                    elif metric == 'anomaly_depth':
                        trees_score[f, t] = avg_outlier_depth
                else:
                    trees_score[f, t] = 0
    
    # Threshold computation
    thresholds = np.zeros(num_forests)
    for f in range(num_forests):
        if mode == 'mean':
            # print("Using mean thresholding")
            thresholds[f] = np.mean(trees_score[f])
        elif mode == 'percentile':  
            # print(f"Using {percentile}th percentile thresholding")
            thresholds[f] = np.percentile(trees_score[f], percentile)
    
    pruned_forests = remove_trees(iforests, trees_score, thresholds)
    return pruned_forests

In [18]:
def post_pruned_evaluation(X_test:np.ndarray, y_test:np.ndarray, pruned_if:IsolationForest):
    """
    Evaluate the pruned Isolation Forest models on the test set.
    Args:
        X_test (np.ndarray): Test data.
        y_test (np.ndarray): True labels for the test data.
        pruned_if (list): List of pruned Isolation Forest models.
    Returns:
        f1s (np.ndarray): Array of F1-scores for each pruned model.
        avps (np.ndarray): Array of Average Precision scores for each pruned model.
    """
    
    f1s, avps = [], []

    for f in range(len(pruned_if)):
        # get predictions
        y_pred = pruned_if[f].predict(X_test)
        y_pred = np.where(y_pred == -1, 1, 0)  # map -1 to 1 (anomalies), 1 to 0 (inliers)
        anomaly_scores = 0.5 * (-pruned_if[f].decision_function(X_test) + 1)
        # compute performance metrics
        f1 = f1_score(y_test, y_pred)
        avg_precision = average_precision_score(y_test, anomaly_scores)

        f1s.append(f1)
        avps.append(avg_precision)

    return np.asarray(f1s), np.asarray(avps)

In [19]:
def get_depth_in_out(iforests, depth_train, y_train):
    
    num_forests = len(iforests)
    num_trees = len(iforests[0].estimators_)

    outlier_depths = [[None for _ in range(num_trees)] for _ in range(num_forests)]
    inlier_depths = [[None for _ in range(num_trees)] for _ in range(num_forests)]
    
    for f in range(num_forests):
        for t in range(len(iforests[f].estimators_)):
            # Extract depth values (skip None values)
            tree_depths = depth_train[f][t]
            valid_indices = np.array([i for i in range(len(tree_depths)) if tree_depths[i] is not None])
            
            if len(valid_indices) > 0:
                valid_depths = np.array([tree_depths[i] for i in valid_indices], dtype=float)
                valid_labels = y_train[valid_indices]
                
                # Check both classes presence
                if np.sum(valid_labels) > 0 and np.sum(valid_labels == 0) > 0:
                    outlier_depths[f][t] = valid_depths[valid_labels == 1]
                    inlier_depths[f][t] = valid_depths[valid_labels == 0]

    return np.asarray(outlier_depths), np.asarray(inlier_depths)

In [20]:
def log_depth_distributions(outlier_depths, inlier_depths, seed):
    """
    Log depth distributions for outliers and inliers to wandb.
    
    Args:
        outlier_depths (list): Nested list of outlier depths [forest][tree].
        inlier_depths (list): Nested list of inlier depths [forest][tree].
        seed (int): Random seed used for the models.
    """

    all_outlier_depths = np.concatenate([
        depth_array for forest in outlier_depths 
        for depth_array in forest 
        if depth_array is not None and len(depth_array) > 0
    ])
    
    all_inlier_depths = np.concatenate([
        depth_array for forest in inlier_depths 
        for depth_array in forest 
        if depth_array is not None and len(depth_array) > 0
    ])

    mean_outlier = np.mean(all_outlier_depths)
    mean_inlier = np.mean(all_inlier_depths)

    plt.figure(figsize=(14, 7))
    bins = np.linspace(0.1, 0.5, 30) 
    plt.hist(all_outlier_depths, bins=bins, density=True, color='red', alpha=0.6, label='Anomalous Points')
    plt.hist(all_inlier_depths, bins=bins, density=True, color='blue', alpha=0.4, label='Normal Points')
    plt.axvline(mean_outlier, color='red', linestyle='--', linewidth=2, alpha=0.8, 
                label=f'Anomalous Mean: {mean_outlier:.3f}')
    plt.axvline(mean_inlier, color='blue', linestyle='--', linewidth=2, alpha=0.8, 
                label=f'Normal Mean: {mean_inlier:.3f}')
    plt.title('Histogram of Depth Stats', fontsize=16)
    plt.xlabel('Inverse Depth', fontsize=12)
    plt.ylabel('Density', fontsize=12)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

In [21]:
def plot_estimators_depth_distribution_2(outlier_depths, inlier_depths, unique_usage, forest_idx, percentile, seed, threshold_type):
    """
    For a single forest, plots the depth distribution for each tree.
    
    Args:
        outlier_depths (list): Nested list of outlier depths [forest][tree].
        inlier_depths (list): Nested list of inlier depths [forest][tree].
        unique_usage (np.ndarray): Array of shape (num_forests, num_trees) containing unique usage per tree.
        forest_idx (int): The index of the forest to inspect.
        percentile (int): Percentile value for threshold.
        seed (int): The random seed for labeling.
        threshold_type (str): Type of threshold to apply.
    """

    num_trees = len(outlier_depths[forest_idx])

    separation_scores = []
    for t in range(num_trees):
        outlier_tree_depths = outlier_depths[forest_idx][t]
        inlier_tree_depths = inlier_depths[forest_idx][t]
        
        if outlier_tree_depths is not None and inlier_tree_depths is not None and len(outlier_tree_depths) > 0 and len(inlier_tree_depths) > 0:
            mean_diff = np.mean(outlier_tree_depths) / np.mean(inlier_tree_depths)
            separation_scores.append(mean_diff)
        else:
            separation_scores.append(0) 

    threshold_label = ''

    if threshold_type == 'mean':
        threshold = np.mean(separation_scores)
        threshold_label = 'Mean'
    elif threshold_type == 'percentile':
        threshold = np.percentile(separation_scores, percentile)
        threshold_label = f'{percentile}th Percentile'

    # First plot: Bar plot with separation scores and unique usage
    plt.figure(figsize=(15, 7))
    
    # Extract unique usage for the selected forest
    forest_unique_usage = unique_usage[forest_idx]
    
    # Create dual y-axis plot
    fig, ax1 = plt.subplots(figsize=(15, 7))
    
    x_pos = np.arange(num_trees)
    
    # Plot separation scores on primary axis
    color1 = 'teal'
    ax1.set_xlabel('Tree Index', fontsize=12)
    ax1.set_ylabel('Separation Score [Mean(Anomalous Depth) / Mean(Normal Depth)]', color=color1, fontsize=12)
    bars1 = ax1.bar(x_pos - 0.2, separation_scores, width=0.4, color=color1, alpha=0.7, label='Separation Score')
    ax1.tick_params(axis='y', labelcolor=color1)
    ax1.axhline(0, color='black', linestyle='--', linewidth=1)
    ax1.grid(True, linestyle='--', axis='y', alpha=0.6)
    
    # Create secondary y-axis for unique usage
    ax2 = ax1.twinx()
    color2 = 'orange'
    ax2.set_ylabel('Unique Usage', color=color2, fontsize=12)
    bars2 = ax2.bar(x_pos + 0.2, forest_unique_usage, width=0.4, color=color2, alpha=0.7, label='Unique Usage')
    ax2.tick_params(axis='y', labelcolor=color2)
    
    # Add legends
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=12)
    
    plt.title(f'Tree Separation Score and Unique Usage (Forest {forest_idx} - Seed {seed})', fontsize=16)
    plt.tight_layout()
    plt.show()

    # Second plot: Histogram of separation scores
    plt.figure(figsize=(10, 6))
    ax = sns.histplot(separation_scores, kde=True)

    # Add numeric values on top of each bar
    for patch in ax.patches:
        height = patch.get_height()
        if height > 0:  # Only add label if bar has height
            ax.text(patch.get_x() + patch.get_width() / 2., height,
                    f'{int(height)}',
                    ha='center', va='bottom', fontsize=9)

    plt.title('Distribution of Tree Quality Scores', fontsize=16)
    plt.xlabel('Separation Score', fontsize=12)
    plt.ylabel('Count of Trees', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6, axis='y')
    plt.show()

In [22]:
def plot_estimators_depth_distribution(outlier_depths, inlier_depths, forest_idx, percentile, seed, threshold_type):
    """
    For a single forest, plots the depth distribution for each tree.
    
    Args:
        outlier_depths (list): Nested list of outlier depths [forest][tree].
        inlier_depths (list): Nested list of inlier depths [forest][tree].
        forest_index (int): The index of the forest to inspect.
        seed (int): The random seed for labeling.
    """

    num_trees = len(outlier_depths[forest_idx])

    separation_scores = []
    for t in range(num_trees):
        outlier_tree_depths = outlier_depths[forest_idx][t]
        inlier_tree_depths = inlier_depths[forest_idx][t]
        
        if outlier_tree_depths is not None and inlier_tree_depths is not None and len(outlier_tree_depths) > 0 and len(inlier_tree_depths) > 0:
            mean_diff = np.mean(outlier_tree_depths) / np.mean(inlier_tree_depths)
            separation_scores.append(mean_diff)
        else:
            separation_scores.append(0) 

    threshold_label = ''

    if threshold_type == 'mean':
        threshold = np.mean(separation_scores)
        threshold_label = 'Mean'
    elif threshold_type == 'percentile':
        threshold = np.percentile(separation_scores, percentile)
        threshold_label = f'{percentile}th Percentile'

    plt.figure(figsize=(15, 7))
    plt.bar(range(len(separation_scores)), separation_scores, color='teal')
    # plt.axhline(threshold, color='red', linestyle='--', linewidth=2, label=f'{threshold_label} Separation Score Threshold')
    plt.legend(fontsize=12)
    plt.title(f'Tree Separation Score (Forest {forest_idx} - Seed {seed})', fontsize=16)
    plt.xlabel('Tree Index', fontsize=12)
    plt.ylabel('Separation Score [Mean(Anomalous Depth) / Mean(Normal Depth)]', fontsize=12)
    plt.axhline(0, color='black', linestyle='--', linewidth=1) 
    plt.grid(True, linestyle='--', axis='y', alpha=0.6)
    plt.show()

    plt.figure(figsize=(10, 6))
    ax = sns.histplot(separation_scores, kde=True)

    # Add numeric values on top of each bar
    for patch in ax.patches:
        height = patch.get_height()
        if height > 0:  # Only add label if bar has height
            ax.text(patch.get_x() + patch.get_width() / 2., height,
                    f'{int(height)}',
                    ha='center', va='bottom', fontsize=9)

    plt.title('Distribution of Tree Quality Scores', fontsize=16)
    plt.xlabel('Separation Score', fontsize=12)
    plt.ylabel('Count of Trees', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6, axis='y')
    plt.show()

# Experiments

## Parameters

In [ ]:
num_trees = 100
max_samples = 256  
num_forests = 10
test_size = 0.2
seeds = np.arange(30).tolist()   
# majority_voting_threshold = 0.5  # Threshold for majority voting
# num_random_runs = 30    # Number of random runs for comparison

In [24]:
threshold_type = 'percentile'      # 'random', 'mean', 'majority_voting', 'percentile' 
selection_method = 'elbow'      # 'elbow', 'soft_threshold'
percentile = 80             # for percentile thresholding
# num_trees_to_remove = 50    # for random removal
metric = 'sparation_ratio' # 'sparation_ratio', 'separation_diff', 'anomaly_depth'

## Experiment loop

In [25]:
# Collect all datasets
datasets = odds_datasets.datasets_names

datasets = [d for d in datasets if d in ['thyroid']]    # , 'vertebral', 'breastw'
# datasets = [d for d in datasets if d not in ['wbc', 'mammography']]

for i, dataset in tqdm.tqdm(enumerate(datasets), desc="Loading datasets", total=len(datasets)):
    # Initialize results list to store the results of each dataset
    results = []

    X,y = odds_datasets.load(dataset)
    # Get the unique features used in the dataset
    all_features = range(X.shape[1])

    print(f"\nProcessing dataset: {dataset}")

    for seed in tqdm.tqdm(seeds, leave=False):

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=seed, stratify=y) if test_size > 0 else (X, X, y, y)
        contamination = np.mean(y_train)
        X_train, y_train = shuffle(X_train, y_train, random_state=seed)

        # run = wandb.init(
        #     project="if_optimization",
        #     name=f"diffi_odds_seed_{seed}",
        #     config={
        #         "seed": seed,
        #         "dataset": dataset,
        #         "num_forests": num_forests,
        #         "num_trees": num_trees,
        #         "contamination": contamination,
        #         "max_samples": max_samples,
        #         "test_size": test_size,
        #         "selection_method": "N/A",          
        #         "thresholds": threshold_type + "_depth_based_" + metric,   
        #         "percentile_value": percentile if threshold_type == 'percentile' else "N/A",            # for percentile thresholding
        #         "num_trees_to_remove": "N/A",    # for random removal
        #         "depth_based_on": "predicted_outliers",  # indicate if depth based on true or predicted outliers
        #     },
        # )

        f1s, avg_precisions, fi_og, iforests, used_features, fis_out, fis_in, y_pred = diffi_ranks(
            X_train,
            X_test,
            y_test,
            seed,
            n_iters=num_forests,
            contamination=contamination,
            num_trees=num_trees,  
        )

        mean_f1 = np.mean(f1s)
        mean_avg_precision = np.mean(avg_precisions)
        # mean_fi_og = np.mean(fi_og, axis=0)

        # selected_features = []
        # if selection_method == 'elbow':
        #     selected_features, gap_thresholds, gaps = elbow_features_selection(fi_og)
        # else:  # 'soft_threshold'
        #     selected_features = [all_features]*num_forests

        # # extract only the feature importances of the selected features
        # unique_fi_og = np.zeros((num_forests, fi_og.shape[1]), dtype=float)
        # for f in range(num_forests):
        #     forest_selected_features = set(selected_features[f])
            
        #     for sf in forest_selected_features:
        #         unique_fi_og[f, sf] = fi_og[f, sf]
        # # normalize the unique feature importances per forest
        # normalized_unique_fi_og = np.divide(unique_fi_og, np.sum(unique_fi_og, axis=1, keepdims=True))
        # # get feature usage
        # usages, normalized_usages = get_feature_usage(used_features, all_features)

        # unique_usages = get_unique_usage_per_tree(normalized_usages, normalized_unique_fi_og, selected_features)

        # # get depths stats for training set
        depth_train = np.array(get_depth_stats(iforests, X_train))

        # outlier_depths, inlier_depths = get_depth_in_out(iforests, depth_train, y_train) 

        # log_depth_distributions(outlier_depths, inlier_depths, seed)  
        # plot_estimators_depth_distribution(outlier_depths, inlier_depths, 0, percentile, seed=seed, threshold_type=threshold_type)
        # plot_estimators_depth_distribution_2(outlier_depths, inlier_depths, unique_usages, 0, percentile, seed=seed, threshold_type=threshold_type)

        # Apply depth-based pruning
        pruned_iforests = depth_based_pruning(
            iforests, 
            depth_train, 
            y_train, 
            mode=threshold_type,
            percentile=percentile, 
            metric=metric
        )
    
        # print(f"Trees indices kept after pruning fneor isolation forest 0 - seed {seed}: {best_trees_idx[0]}")

        pruned_num_trees = [len(pruned_iforest.estimators_) for pruned_iforest in pruned_iforests]

        # log_estimators_summary(pruned_num_trees, num_trees, threshold_type, seed)

        # Evaluate pruned model performance
        f1s_pruned, avg_precisions_pruned = post_pruned_evaluation(
            X_test, 
            y_test, 
            pruned_iforests
        )

        mean_f1_pruned = np.mean(f1s_pruned)
        mean_avg_precision_pruned = np.mean(avg_precisions_pruned)

        results.append([
            seed,
            f"{mean_f1:.4f}",
            f"{mean_f1_pruned:.4f}",
            f"{mean_avg_precision:.4f}",
            f"{mean_avg_precision_pruned:.4f}",
        ])


    headers = ["Seed", "F1 Score", "F1 Score Pruned", "Avg Precision", "Avg Precision Pruned"]
    table = tabulate(results, headers, tablefmt="pretty")
    print(table)

    # log performance metrics to wandb
    # table = wandb.Table(data=results, columns=headers)
    # wandb.log({f"{dataset}_results_table": table})

    print(f"\nDataset: {dataset}")
    print(f"average original F1 score: {np.mean([float(r[1]) for r in results]):.4f} +/- {np.std([float(r[1]) for r in results]):.4f}")
    print(f"average pruned F1 score: {np.mean([float(r[2]) for r in results]):.4f} +/- {np.std([float(r[2]) for r in results]):.4f}")
    print(f"average original Avg Precision: {np.mean([float(r[3]) for r in results]):.4f} +/- {np.std([float(r[3]) for r in results]):.4f}")
    print(f"average pruned Avg Precision: {np.mean([float(r[4]) for r in results]):.4f} +/- {np.std([float(r[4]) for r in results]):.4f}")

Loading datasets:   0%|          | 0/1 [00:00<?, ?it/s]


Processing dataset: thyroid


Loading datasets:   0%|          | 0/1 [00:29<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
# run.finish()